In [1]:
# ============================================================================
# TSFNet -- single-file, notebook-ready version (no relative imports).
# Combines layers.py + model.py + losses.py + dataset.py + train.py from the
# tsfnet_model package into one script, in dependency order, so it can be
# pasted into Jupyter cells (split at the "# %% CELL n" markers below, or
# just run the whole thing as one script/cell).
#
# See the original package's README.md for the full explanation of every
# adaptation made to the paper (3-class output, reduced model size, added
# weight decay, reduced stage1/stage2 epoch budgets).
# ============================================================================

# %% CELL 1 -- imports
import argparse
import copy
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

In [3]:
# %% CELL 2 -- layers.py (Conv3dSame, ConvBlock3D, EEGLikeBranch, FNIRSBranch, EFGFLayer, CAFELayer)
        self.fusion_proj = nn.Linear(fusion_dim, d_model)
        self.fnirs_proj = nn.Linear(fnirs_dim, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, n_segments, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # Cross-attention 1: fNIRS (query) attends to fusion (key/value)
        self.cross_attn_fnirs_to_fusion = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)
        # Cross-attention 2: fusion (query) attends to fNIRS (key/value)
        self.cross_attn_fusion_to_fnirs = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)

        self.fusion_out_proj = nn.Linear(d_model, fusion_dim)
        self.fnirs_out_proj = nn.Linear(d_model, fnirs_dim)

        # Gated fusion (trainable scalar gates, sigmoid-constrained)
        self.gate_fusion = nn.Parameter(torch.zeros(1))
        self.gate_fnirs = nn.Parameter(torch.zeros(1))

    def forward(self, fusion_feat, fnirs_feat):
        """
        fusion_feat: (B, 32, 4, 4)      -- Fusion branch's pooled F_fusion
        fnirs_feat:  (B, 11, 32, 4, 4)  -- fNIRS branch's pooled F_fnirs (per segment)
        returns: enhanced_fusion (B, fusion_dim), enhanced_fnirs (B, fnirs_dim)  [for PCC + classifier]
        """
        B = fusion_feat.shape[0]
        fusion_flat = fusion_feat.reshape(B, -1)                       # (B, 512)
        fnirs_flat = fnirs_feat.reshape(B, self.n_segments, -1)        # (B, 11, 512)

        fusion_tok = self.fusion_proj(fusion_flat).unsqueeze(1)        # (B, 1, d_model)
        fnirs_tok = self.fnirs_proj(fnirs_flat) + self.pos_embed       # (B, 11, d_model)

        # Cross-attention 1: fNIRS attends to fusion
        fnirs_enhanced, _ = self.cross_attn_fnirs_to_fusion(
            query=fnirs_tok, key=fusion_tok, value=fusion_tok)         # (B, 11, d_model)
        # Cross-attention 2: fusion attends to fNIRS
        fusion_enhanced, _ = self.cross_attn_fusion_to_fnirs(
            query=fusion_tok, key=fnirs_tok, value=fnirs_tok)          # (B, 1, d_model)

        fnirs_enhanced_full = self.fnirs_out_proj(fnirs_enhanced)      # (B, 11, 512)
        fnirs_enhanced_full = fnirs_enhanced_full.mean(dim=1)          # pool segments -> (B, 512)
        fusion_enhanced_full = self.fusion_out_proj(fusion_enhanced.squeeze(1))  # (B, 512)

        g_fusion = torch.sigmoid(self.gate_fusion)
        g_fnirs = torch.sigmoid(self.gate_fnirs)
        fnirs_flat_pooled = fnirs_flat.mean(dim=1)                     # original, pooled -> (B, 512)

        enhanced_fusion = g_fusion * fusion_enhanced_full + (1 - g_fusion) * fusion_flat
        enhanced_fnirs = g_fnirs * fnirs_enhanced_full + (1 - g_fnirs) * fnirs_flat_pooled

        return enhanced_fusion, enhanced_fnirs

IndentationError: unindent does not match any outer indentation level (<string>, line 21)